# GolStats - StatsBomb Data Ingestion



1. Configuration
2. Load competitions
3. Select competition
4. Load matches
5. Select matches
6. Save match metadata
7. Download events
8. Validate ingestion

## 1. Configuration

This notebook is responsible for ingesting football data from StatsBomb Open Data.

The ingestion process retrieves competition and match information and downloads
raw event data for selected matches.

The raw JSON files are stored in a Databricks Volume and will be processed
later by the Bronze layer.

Flow:

StatsBomb Open Data
        ↓
Competition metadata
        ↓
Match metadata
        ↓
Raw event JSON
        ↓
Databricks Volume

In [0]:
import requests
import json
from datetime import datetime

### Storage configuration

The raw files will be stored in the Bronze Volume registered in Unity Catalog.

In [0]:
VOLUME_PATH = "/Volumes/golstats/bronze/raw_files"

BASE_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

## 2. Load available competitions

StatsBomb provides a list of competitions and seasons available in its
Open Data repository.

We first retrieve this metadata so the pipeline can dynamically identify
which competition and season we want to ingest.

In [0]:
competitions_url = f"{BASE_URL}/competitions.json"

response = requests.get(competitions_url)

response.raise_for_status()

competitions = response.json()

len(competitions)

80

In [0]:
competitions_df = spark.createDataFrame(competitions)

display(competitions_df)

competition_gender,competition_id,competition_international,competition_name,competition_youth,country_name,match_available,match_available_360,match_updated,match_updated_360,season_id,season_name
male,9,false,1. Bundesliga,false,Germany,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,281,2023/2024
male,9,false,1. Bundesliga,false,Germany,2024-05-19T11:11:14.192381,null,2024-05-19T11:11:14.192381,null,27,2015/2016
male,1267,true,African Cup of Nations,false,Africa,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,107,2023
male,16,false,Champions League,false,Europe,2026-05-15T15:54:04.598614,null,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,4,2018/2019
male,16,false,Champions League,false,Europe,2024-02-13T02:35:28.134882,null,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,1,2017/2018
male,16,false,Champions League,false,Europe,2024-02-13T02:37:32.205154,null,2024-02-13T02:37:32.205154,2021-06-13T16:17:31.694,2,2016/2017
male,16,false,Champions League,false,Europe,2024-06-12T07:45:38.786894,null,2024-06-12T07:45:38.786894,2021-06-13T16:17:31.694,27,2015/2016
male,16,false,Champions League,false,Europe,2024-02-12T12:49:54.914228,null,2024-02-12T12:49:54.914228,2021-06-13T16:17:31.694,26,2014/2015
male,16,false,Champions League,false,Europe,2024-02-12T12:48:48.479157,null,2024-02-12T12:48:48.479157,2021-06-13T16:17:31.694,25,2013/2014
male,16,false,Champions League,false,Europe,2024-02-12T12:47:34.340413,null,2024-02-12T12:47:34.340413,2021-06-13T16:17:31.694,24,2012/2013


## 3. Select competition and season

For this project we will analyze the FIFA World Cup 2022.

Competition:
- Name: FIFA World Cup
- Competition ID: 43
- Season ID: 106
- Season: 2022

Using explicit IDs makes the ingestion reproducible and avoids relying on
the position of a record in the competitions dataset.

In [0]:
COMPETITION_ID = 43
SEASON_ID = 106

print(f"Competition ID: {COMPETITION_ID}")
print(f"Season ID: {SEASON_ID}")

Competition ID: 43
Season ID: 106


In [0]:
competitions_df.filter(
    (competitions_df.competition_id == COMPETITION_ID) &
    (competitions_df.season_id == SEASON_ID)
).select(
    "competition_id",
    "competition_name",
    "season_id",
    "season_name"
).show(truncate=False)

+--------------+----------------+---------+-----------+
|competition_id|competition_name|season_id|season_name|
+--------------+----------------+---------+-----------+
|43            |FIFA World Cup  |106      |2022       |
+--------------+----------------+---------+-----------+



## 4. Load match metadata

Once the competition and season are selected, we retrieve the list of
matches belonging to that competition.

At this stage we are not downloading event data yet.

We first obtain the match metadata, including the match ID, date and
participating teams.

The match ID will later be used to retrieve the event data for each match.

In [0]:
matches_url = (
    f"{BASE_URL}/matches/"
    f"{COMPETITION_ID}/"
    f"{SEASON_ID}.json"
)

response = requests.get(matches_url)

response.raise_for_status()

matches = response.json()

print(f"Total matches available: {len(matches)}")

Total matches available: 64


In [0]:
# Convertimos únicamente los campos que necesitamos para el proyecto.
# Evitamos que Spark tenga que inferir automáticamente todas las estructuras
# anidadas presentes en el JSON de StatsBomb.

matches_clean = []

for match in matches:
    matches_clean.append({
        "match_id": match.get("match_id"),
        "match_date": match.get("match_date"),
        "home_team": match.get("home_team", {}).get("home_team_name"),
        "away_team": match.get("away_team", {}).get("away_team_name")
    })

matches_df = spark.createDataFrame(matches_clean)

display(matches_df)

away_team,home_team,match_date,match_id
Morocco,Canada,2022-12-01,3857276
Iran,England,2022-11-21,3857271
Belgium,Croatia,2022-12-01,3857296
Ecuador,Netherlands,2022-11-25,3857274
Spain,Japan,2022-12-01,3857255
United States,England,2022-11-25,3857272
United States,Iran,2022-11-29,3857278
Croatia,Morocco,2022-11-23,3857277
Iran,Wales,2022-11-25,3857273
France,Tunisia,2022-11-30,3857275


## 5. Select matches for ingestion

The FIFA World Cup 2022 contains 64 matches.

For the initial development phase, we will ingest a smaller subset of 10 matches.
This allows us to validate the ingestion and Bronze pipeline before processing
the complete competition dataset.

The matches are ordered by date to make the selection deterministic and
reproducible.

The match ID will be preserved because it is the key used to associate each
event with its corresponding match.

In [0]:
# Ordenamos los partidos por fecha para que la selección sea reproducible.
selected_matches_df = (
    matches_df
    .orderBy("match_date", "match_id")
    .limit(10)
)

display(selected_matches_df)

away_team,home_team,match_date,match_id
Ecuador,Qatar,2022-11-20,3857286
Iran,England,2022-11-21,3857271
Wales,United States,2022-11-21,3857282
Netherlands,Senegal,2022-11-21,3857285
Tunisia,Denmark,2022-11-22,3857254
Poland,Mexico,2022-11-22,3857265
Australia,France,2022-11-22,3857279
Saudi Arabia,Argentina,2022-11-22,3857300
Canada,Belgium,2022-11-23,3857268
Croatia,Morocco,2022-11-23,3857277


### Extract match IDs

The match ID uniquely identifies each game in the StatsBomb dataset.

We will use these IDs to download the corresponding event JSON files.

In [0]:
# Extraemos los IDs de los partidos seleccionados.
match_ids = [
    row["match_id"]
    for row in selected_matches_df.select("match_id").collect()
]

print(f"Matches selected: {len(match_ids)}")
print(match_ids)

Matches selected: 10
[3857286, 3857271, 3857282, 3857285, 3857254, 3857265, 3857279, 3857300, 3857268, 3857277]


## 6. Save match metadata

In addition to the raw event data, we store the metadata of the selected
matches.

This metadata will allow us to identify the teams involved in each match
and will later be used to enrich the Silver and Gold layers.

The original raw event data remains unchanged.

In [0]:
# Definimos la ubicación donde almacenaremos los metadatos.
MATCH_METADATA_PATH = f"{VOLUME_PATH}/matches_metadata.json"

# Convertimos los registros seleccionados a una lista de diccionarios.
metadata = [
    row.asDict()
    for row in selected_matches_df.collect()
]

# Guardamos los metadatos como JSON.
with open(MATCH_METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Match metadata saved to: {MATCH_METADATA_PATH}")

Match metadata saved to: /Volumes/golstats/bronze/raw_files/matches_metadata.json


## 7. Download raw event data

For each selected match, we retrieve the complete event dataset from
StatsBomb Open Data.

Each match is stored as an individual JSON file in the Databricks Volume.

At this stage, the data is kept in its original raw format.

No cleaning, transformation, aggregation or business logic is applied here.

These transformations will be performed in the Bronze, Silver and Gold layers.

In [0]:
# Download the raw event data for each selected match.

for match_id in match_ids:

    # URL containing the event data for the current match.
    events_url = f"{BASE_URL}/events/{match_id}.json"

    # Request the JSON file from StatsBomb.
    response = requests.get(events_url)
    response.raise_for_status()

    # Define the destination path in the Databricks Volume.
    file_path = f"{VOLUME_PATH}/eventos_{match_id}.json"

    # Save the raw JSON without modifying its contents.
    with open(file_path, "w") as f:
        f.write(response.text)

    print(f"Match {match_id} ingested successfully")

Match 3857286 ingested successfully
Match 3857271 ingested successfully
Match 3857282 ingested successfully
Match 3857285 ingested successfully
Match 3857254 ingested successfully
Match 3857265 ingested successfully
Match 3857279 ingested successfully
Match 3857300 ingested successfully
Match 3857268 ingested successfully
Match 3857277 ingested successfully


## 8. Validate ingestion

Before processing the data, we verify that the expected raw event files
were successfully created in the Databricks Volume.

This validation helps detect failed downloads before the data enters
the Bronze layer.

In [0]:
# List all files currently stored in the raw data Volume.
files = dbutils.fs.ls(VOLUME_PATH)

display(files)

path,name,size,modificationTime
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857254.json,eventos_3857254.json,3128208,1788463454000
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857265.json,eventos_3857265.json,2702580,1788463454000
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857268.json,eventos_3857268.json,2990522,1788463456000
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857271.json,eventos_3857271.json,3063347,1788463451000
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857277.json,eventos_3857277.json,3090039,1788463456000
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,eventos_3857279.json,3354655,1788463455000
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857282.json,eventos_3857282.json,3081742,1788463452000
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857285.json,eventos_3857285.json,2689430,1788463453000
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857286.json,eventos_3857286.json,2799478,1788463451000
dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857300.json,eventos_3857300.json,2865394,1788463455000


In [0]:
# Validate that all selected matches were successfully downloaded.

downloaded_match_ids = [
    int(file.name.replace("eventos_", "").replace(".json", ""))
    for file in files
    if file.name.startswith("eventos_")
]

# Check which selected matches are present in the Volume.
missing_matches = [
    match_id
    for match_id in match_ids
    if match_id not in downloaded_match_ids
]

print(f"Selected matches: {len(match_ids)}")
print(f"Downloaded selected matches: {len(match_ids) - len(missing_matches)}")
print(f"Missing matches: {len(missing_matches)}")

if missing_matches:
    print("Missing match IDs:", missing_matches)
else:
    print("All selected matches were successfully ingested.")

Selected matches: 10
Downloaded selected matches: 10
Missing matches: 0
All selected matches were successfully ingested.
